In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

db_file_nm = 'clarity_analytics_center.db'
db_file_dir  = Path.cwd().parent.joinpath('storage/')
db_path = Path(db_file_dir).joinpath(db_file_nm)
    
db_path.parent.mkdir(parents=True, exist_ok=True)
if db_path.exists():
    print(f"Database file '{db_path}' already exists. removing and recreating")
    db_path.unlink(missing_ok=True)


# Create or connect to the database
conn = sqlite3.connect(db_path)

# Enable SQLite foreign-key support
conn.execute("PRAGMA foreign_keys = ON;")


Database file 'c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\clarity_analytics_center.db' already exists. removing and recreating


# BRONZE

In [2]:
# ICE operations table: bronze_ice_operations
conn.execute("""
CREATE TABLE IF NOT EXISTS bronze_ice_operations
(
    agent_id                   INTEGER NOT NULL,
    region                     TEXT NOT NULL,
    role                       TEXT NOT NULL,
    department                 TEXT NOT NULL,
    unit                       TEXT NOT NULL,
    assignment_start_date      TEXT NOT NULL,
    assignment_end_date        TEXT,
    detention_center_name      TEXT NOT NULL,
    supervisor_id              INTEGER
);
""")

# ICE budget table: bronze_ice_budget
conn.execute("""
CREATE TABLE IF NOT EXISTS bronze_ice_budget
(
    fiscal_year                      INTEGER NOT NULL,
    department_code                  TEXT NOT NULL,
    agency_name                      TEXT NOT NULL,
    net_operating_cost               REAL,
    budget_authority                 REAL,
    budget_deficit_contribution      REAL,
    source                           TEXT NOT NULL
);
""")

# Treasury reconciliation table: bronze_treasury_reconciliation
conn.execute("""
CREATE TABLE IF NOT EXISTS bronze_treasury_reconciliation
(
    record_date                  TEXT,
    stmt_fiscal_year             TEXT,
    restmt_flag                  TEXT,
    account_desc                 TEXT,
    component_desc               TEXT,
    line_item_desc               TEXT,
    position_bil_amt             TEXT,
    src_line_nbr                 TEXT,
    record_fiscal_year           TEXT,
    record_fiscal_quarter        TEXT,
    record_calendar_year         TEXT,
    record_calendar_quarter      TEXT,
    record_calendar_month        TEXT,
    record_calendar_day          TEXT
);
""")

# ICE enforcement table: bronze_ice_enforcement_pdfs
conn.execute("""
CREATE TABLE IF NOT EXISTS bronze_ice_enforcement_pdfs
(
    extraction_timestamp       TEXT NOT NULL,
    source_file                TEXT NOT NULL,
    record_count               INTEGER NOT NULL,
    extraction_status          TEXT NOT NULL,
    page_number                INTEGER NOT NULL,
    content                    TEXT
);
""")

# ICE enforcement table: bronze_ice_enforcement_metrics
conn.execute("""
CREATE TABLE IF NOT EXISTS bronze_ice_enforcement_metrics
(
    source_file                TEXT NOT NULL,
    report_year                TEXT NOT NULL,
    content_year               TEXT NOT NULL,
    page_number                INTEGER,
    metric_name                TEXT,
    metric_value               REAL,
    metric_comment	           TEXT
);
""")

In [3]:
# commit changes and show created tables
conn.commit()
created_tables_df = pd.read_sql_query(f'SELECT name FROM sqlite_master WHERE type="table" '
                    f'AND name != "sqlite_sequence"', conn)
print(created_tables_df)

                             name
0           bronze_ice_operations
1               bronze_ice_budget
2  bronze_treasury_reconciliation
3     bronze_ice_enforcement_pdfs
4  bronze_ice_enforcement_metrics


# SILVER

In [4]:
# silver_operations
conn.execute("""
CREATE TABLE IF NOT EXISTS silver_operations (
    operations_row_id               INTEGER PRIMARY KEY,
    agent_id                        INTEGER,
    supervisor_id                   INTEGER,
    region                          TEXT NOT NULL,
    role                            TEXT NOT NULL,
    department                      TEXT NOT NULL,
    unit                            TEXT NOT NULL,
    detention_center_name           TEXT NOT NULL,
    assignment_start_date           TEXT NOT NULL,
    assignment_end_date             TEXT,
    is_active_assignment            INTEGER NOT NULL,
    silver_processed_timestamp      TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CHECK (is_active_assignment IN (0, 1))
);
""")

# silver_budget
conn.execute("""
CREATE TABLE IF NOT EXISTS silver_budget (
    budget_row_id                   TEXT PRIMARY KEY,
    fiscal_year                     INTEGER NOT NULL,
    department_code                 TEXT NOT NULL,
    agency_name                     TEXT NOT NULL,
    net_operating_cost              REAL,
    budget_authority                REAL,
    budget_deficit_contribution     REAL,
    source                          TEXT NOT NULL,
    silver_processed_timestamp      TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);
""")

# silver_treasury_reconciliation
conn.execute("""
CREATE TABLE IF NOT EXISTS silver_treasury_reconciliation (
    treasury_reconciliation_row_id      TEXT PRIMARY KEY,
    record_date                         TEXT    NOT NULL,
    statement_fiscal_year               INTEGER NOT NULL,
    restatement_flag                    TEXT,
    account_description                 TEXT,
    component_description               TEXT,
    line_item_description               TEXT,
    position_billion_amount             REAL,
    source_line_number                  INTEGER,
    record_fiscal_year                  INTEGER,
    record_fiscal_quarter               INTEGER,
    record_calendar_year                INTEGER,
    record_calendar_quarter             INTEGER,
    record_calendar_month               INTEGER,
    record_calendar_day                 INTEGER,
    silver_processed_timestamp          TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (record_date, source_line_number, restatement_flag)
);
""")

# silver_enforcement
conn.execute("""
CREATE TABLE IF NOT EXISTS silver_enforcement (
    enforcement_row_id              TEXT PRIMARY KEY,
    extraction_timestamp            TEXT NOT NULL,
    source_file                     TEXT NOT NULL,
    page_number                     INTEGER NOT NULL,
    content                         TEXT,
    report_year                     INTEGER,
    metric_name                     TEXT,
    metric_value                    REAL,
    metric_comment                  TEXT,
    silver_processed_timestamp      TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);
""")

In [5]:
# commit changes and show created tables
conn.commit()
created_tables_df = pd.read_sql_query(f'SELECT name FROM sqlite_master WHERE type="table" '
                    f'AND name != "sqlite_sequence"', conn)
print(created_tables_df)

                             name
0           bronze_ice_operations
1               bronze_ice_budget
2  bronze_treasury_reconciliation
3     bronze_ice_enforcement_pdfs
4  bronze_ice_enforcement_metrics
5               silver_operations
6                   silver_budget
7  silver_treasury_reconciliation
8              silver_enforcement


# GOLD

In [6]:


conn.execute("PRAGMA foreign_keys = ON;")

# DIMENSIONS

conn.execute("""
CREATE TABLE IF NOT EXISTS gold_dim_date (
    date_key            INTEGER NOT NULL PRIMARY KEY,   -- yyyymmdd
    full_date           TEXT    NOT NULL UNIQUE,
    calendar_year       INTEGER NOT NULL,
    calendar_quarter    INTEGER NOT NULL,
    calendar_month      INTEGER NOT NULL,
    calendar_day        INTEGER NOT NULL,
    month_name          TEXT    NOT NULL,
    day_name            TEXT    NOT NULL,
    fiscal_year         INTEGER NOT NULL,   -- US federal FY, starts 1 Oct
    fiscal_quarter      INTEGER NOT NULL,
    fiscal_year_label   TEXT    NOT NULL,
    month_end_date      TEXT    NOT NULL,
    is_month_end        INTEGER NOT NULL,
    is_fiscal_year_end  INTEGER NOT NULL,
    is_weekend          INTEGER NOT NULL
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS gold_dim_org_unit (
    org_unit_key    TEXT NOT NULL PRIMARY KEY,   -- md5(department_name, unit_name)
    department_name TEXT NOT NULL,
    unit_name       TEXT NOT NULL,
    org_unit_label  TEXT NOT NULL                -- "department — unit", for display
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS gold_dim_role (
    role_key       TEXT    NOT NULL PRIMARY KEY, -- md5(role_name)
    role_name      TEXT    NOT NULL,
    seniority_rank INTEGER NOT NULL,
    is_leadership  INTEGER NOT NULL
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS gold_dim_facility (
    facility_key   TEXT NOT NULL PRIMARY KEY,    -- md5(facility_name, state_code)
    facility_name  TEXT NOT NULL,
    state_code     TEXT,
    facility_type  TEXT NOT NULL,
    facility_label TEXT NOT NULL                 -- "name (ST)", matches the source
);
""")


conn.execute("""
CREATE TABLE IF NOT EXISTS gold_dim_treasury_line (
    treasury_line_key     TEXT    NOT NULL PRIMARY KEY,  -- md5(component, line_item)
    account_description   TEXT    NOT NULL,
    component_description TEXT    NOT NULL,
    line_item_description TEXT    NOT NULL,
    reconciliation_side   TEXT    NOT NULL,
    line_level            TEXT    NOT NULL,              -- DETAIL / SUBTOTAL / TOTAL
    is_additive           INTEGER NOT NULL               -- 1 when line_level = 'DETAIL'
);
""")




# FACTS

# budget fact: one row per silver_budget row
conn.execute("""
CREATE TABLE IF NOT EXISTS gold_fact_budget (
    budget_key                  TEXT    NOT NULL PRIMARY KEY,
    date_key                    INTEGER NOT NULL,      -- fiscal year end, 30 Sep
    fiscal_year                 INTEGER NOT NULL,
    department_code             TEXT    NOT NULL,
    net_operating_cost          REAL,
    budget_authority            REAL,
    budget_deficit_contribution REAL,
    accrual_to_cash_gap         REAL,                  -- additive difference
    FOREIGN KEY (date_key) REFERENCES gold_dim_date(date_key)
);
""")

# treasury fact : statement fiscal year x line item x restatement flag.
conn.execute("""
CREATE TABLE IF NOT EXISTS gold_fact_treasury (
    treasury_key            TEXT    NOT NULL PRIMARY KEY,
    treasury_line_key       TEXT    NOT NULL,
    date_key                INTEGER NOT NULL,   -- record date
    statement_fiscal_year   INTEGER NOT NULL,
    record_fiscal_year      INTEGER NOT NULL,
    restatement_flag        TEXT    NOT NULL,
    is_restated             INTEGER NOT NULL,
    source_line_number      INTEGER,
    position_billion_amount REAL,
    FOREIGN KEY (treasury_line_key) REFERENCES gold_dim_treasury_line(treasury_line_key),
    FOREIGN KEY (date_key)          REFERENCES gold_dim_date(date_key)
);
""")

# enforcement fact: one row per reported enforcement figure (fiscal year x metric).
conn.execute("""
CREATE TABLE IF NOT EXISTS gold_fact_enforcement (
    enforcement_key TEXT    NOT NULL PRIMARY KEY,
    date_key        INTEGER NOT NULL,           -- fiscal year end, 30 Sep
    fiscal_year     INTEGER NOT NULL,
    metric_name     TEXT    NOT NULL,
    metric_value    REAL    NOT NULL,
    metric_comment  TEXT,
    source_page     INTEGER,
    FOREIGN KEY (date_key) REFERENCES gold_dim_date(date_key)
);
""")

# assignment fact: one row per agent assignment
conn.execute("""
CREATE TABLE IF NOT EXISTS gold_fact_assignment (
    assignment_key        TEXT    NOT NULL PRIMARY KEY,
    start_date_key        INTEGER NOT NULL,
    end_date_key          INTEGER,              -- NULL while the assignment is open
    org_unit_key          TEXT    NOT NULL,
    role_key              TEXT    NOT NULL,
    facility_key          TEXT    NOT NULL,
    agent_id              INTEGER NOT NULL,
    supervisor_id         INTEGER,
    region_name           TEXT    NOT NULL,
    start_date            TEXT    NOT NULL,
    end_date              TEXT,
    start_fiscal_year     INTEGER NOT NULL,
    end_fiscal_year       INTEGER,
    is_active_assignment  INTEGER NOT NULL,
    is_future_dated_end   INTEGER NOT NULL,
    is_self_supervised    INTEGER NOT NULL,
    completed_tenure_days INTEGER,              -- immutable, NULL while open
    FOREIGN KEY (start_date_key) REFERENCES gold_dim_date(date_key),
    FOREIGN KEY (end_date_key)   REFERENCES gold_dim_date(date_key),
    FOREIGN KEY (org_unit_key)   REFERENCES gold_dim_org_unit(org_unit_key),
    FOREIGN KEY (role_key)       REFERENCES gold_dim_role(role_key),
    FOREIGN KEY (facility_key)   REFERENCES gold_dim_facility(facility_key)
);
""")

In [7]:
# commit changes and show created tables
conn.commit()
created_tables_df = pd.read_sql_query(f'SELECT name FROM sqlite_master WHERE type="table" '
                    f'AND name != "sqlite_sequence"', conn)
print(created_tables_df)

                              name
0            bronze_ice_operations
1                bronze_ice_budget
2   bronze_treasury_reconciliation
3      bronze_ice_enforcement_pdfs
4   bronze_ice_enforcement_metrics
5                silver_operations
6                    silver_budget
7   silver_treasury_reconciliation
8               silver_enforcement
9                    gold_dim_date
10               gold_dim_org_unit
11                   gold_dim_role
12               gold_dim_facility
13          gold_dim_treasury_line
14                gold_fact_budget
15              gold_fact_treasury
16           gold_fact_enforcement
17            gold_fact_assignment


In [8]:
conn.close()